# Pipeline hybrid di iPhone: static ALL + exact-length CPU tails

Window encoder penuh 30/120 detik memakai model statis `all`; sisa window memakai model dinamis `cpuOnly` dengan panjang asli. Frontend dan head tetap CPU. Tidak melakukan zero padding atau perubahan anotasi. Bandingkan label dan probabilitas dengan referensi Mac yang sama.

Mulai dengan **Broken Vessels**, yang sebelumnya menyelesaikan benchmark CPU dalam 95,45 detik (pemuatan awal 2,80 detik), dengan kecocokan label Mac 100%. Notebook menyiapkan proyek baru agar baseline CPU tetap tersedia.

Persiapan menghasilkan empat paket statis tambahan, diperiksa paritas CPU sebelum kompilasi Xcode. Paket dinamis ikut dibundel untuk tail; bobot digandakan untuk benchmark dan belum merupakan keputusan kemasan produksi. Sediakan beberapa GiB disk. Gunakan kernel `.venv-songformer311` dari root Section-Annotations.

Decoding/resampling MP3 masih di Mac. Boundary Apple disuplai dari hasil sebelumnya; MusicUnderstanding tidak dijalankan ulang di iPhone.

In [ ]:
from pathlib import Path
from datetime import datetime
import json, subprocess, sys, hashlib
assert Path('finetuning/config.json').is_file()
SONGMAP_ROOT=Path('/Users/danendra/Documents/ADA26/C6/SongMap')
reports=sorted(Path('outputs.local').glob('muq-compression-*/palette6-all-validation.json'),key=lambda p:p.stat().st_mtime,reverse=True)
assert reports,'Validation MuQ palette6 belum ditemukan'
validation=reports[0]
v=json.loads(validation.read_text())
combination=json.loads(Path(v['baseline_combination']).read_text())
pipeline=json.loads(Path(combination['encoder_pipeline_report']).read_text())
expected=pipeline['musicfm_override']['report_sha256']
proofs=[p for p in Path('outputs.local').glob('musicfm-compression-*/palette6/compression_report.json')
        if hashlib.sha256(p.read_bytes()).hexdigest()==expected]
assert len(proofs)==1,'Isi musicfm_proof secara manual dengan report kompresi MusicFM yang sesuai'
musicfm_proof=proofs[0]
output=Path('outputs.local')/('pipeline-hybrid-iphone-'+datetime.now().strftime('%Y%m%d-%H%M%S'))
# Kosong = lagu validation pertama. Isi track_id untuk lagu lain setelah pengujian pertama.
TRACK_ID='Broken Vessels (Amazing Grace)/0304 - Broken Vessels (Amazing Grace).mp3'
print('MuQ validation:',validation)
print('MusicFM proof:',musicfm_proof)
print('Output:',output)


In [ ]:
command=[sys.executable,'-u','-m','deployment.prepare_pipeline_iphone',
         '--songmap-root',str(SONGMAP_ROOT),'--muq-validation',str(validation),
         '--musicfm-compression-report',str(musicfm_proof),'--output-dir',str(output),'--hybrid']
if TRACK_ID:
    command+=['--track-id',TRACK_ID]
log=output.with_suffix('.prepare.log')
log.parent.mkdir(parents=True,exist_ok=True)
with log.open('w',encoding='utf-8') as handle:
    process=subprocess.Popen(command,stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True,bufsize=1)
    try:
        for line in process.stdout:
            print(line,end='');handle.write(line);handle.flush()
        status=process.wait()
    except KeyboardInterrupt:
        process.terminate();process.wait();raise
if status:
    raise RuntimeError(f'Persiapan gagal ({status}); lihat {log.resolve()}')
subprocess.run(['open',str(output/'Benchmark.xcodeproj')],check=True)


## Pengujian

1. Pilih Signing Team dan iPhone fisik di Xcode, lalu jalankan aplikasi.
2. Tekan **Run full pipeline** dan biarkan aplikasi tetap terbuka. Progres `ROUTE` menunjukkan `all_static` atau `cpuOnly_dynamic_tail`.
3. Bagikan `pipeline-benchmark-*.json`. Utamakan `section_agreement_with_mac`, `max_section_probability_error`, `low_coverage_sections`, dan waktu total `total_load_and_pcm_pipeline_seconds`.
4. Untuk perbandingan lebih baik, buka ulang aplikasi dan ulangi pengujian; panggilan pertama bisa mencakup persiapan runtime yang lebih lama. Jangan menganggapnya cold-start terkontrol.

`pcm_pipeline_seconds` mencakup bridge Swift, inferensi, pooling, dan pemuatan encoder selama pergantian jalur. Pemuatan frontend/head awal tercatat pada `model_load_seconds`; jumlah keduanya ada di `total_load_and_pcm_pipeline_seconds`. Waktu prediksi tiap komponen belum mencakup semua pekerjaan Swift tersebut.

`encoder_windows` mencatat jalur setiap window, dan `encoder_load_events` mencatat biaya pemuatan ulang. Snapshot memori bukan peak memory. Mode `all` tidak membuktikan penempatan pada ANE. Jika assert/crash, kirim `ROUTE`/`ENCODER_LOAD`/`PREDICT_START` terakhir dari Xcode. Belum otomatis disetujui untuk integrasi SongMap.